# Problem 3 — Playbook Architecture 

## High-Level Architecture

This graph explains how it handles the state flow, loop(agent loop) and updates the Observer layer to render updated states

![My Image](p3-architecture.jpg)

---------------------------------------------------------------------------------

## SUGGESTION (IMPORTANT)

One of the most important traits in the system above is timely update of "state" and clear state/context management. In this case, LangGraph is the best usecase that is suitable for the exact system.

### Upsides:
- Easier and more explicit state updates and flow(state transitions) as state is more explicitly shared and managed in LangGraph
- Easier to debug within the explicit state flow

### Downsides:
- The application can be heavier as an external framework is added under the system
  (So LangGraph can be partially used for certain agentic flow that needs clear state handling)

------------------------------------------------------------------------

## Prototype/Demo Code 

Note: This demo code was designed for prototyping purposes. Specific implementations and bolierplate code are skipped (e.g. LLM API calls, prompt formatting, retry logic etc)

### Skipping Context Observer and Observer Reasoning Layer 
As they are covered in Question 1/2

### Playbook State Manager

In [8]:
class PlaybookManager:
    def __init__(self, hypothesis):
        self.hypothesis: str = hypothesis
        self.insight: str = None
        self.confidence: int = None
        self.outreach_pitch: str = None
        self.feedback: str = None
        

    def provide_state(self):
        """
        Provides the Play loop with corresponding states
        """
        return self.hypothesis

    def update_state(self, key, value):
        """
        Update shared Playbook state

        Args:
            key: the key name of dynamic attribute passed for state update
            value: the value of dynamic attribute passed for state update
        """
        # update state value dynamically
        setattr(self, key, value)

    def get_shared_state(self):
        """
        Pass states to Observer layer to render on UI
        """

        return {
            "hypothesis": self.hypothesis,
            "insight": self.insight,
            "confidence": self.confidence,
            "outreach_pitch": self.outreach_pitch,
            "feedback": self.feedback
        }

### Playbook Modules

#### Parent Class for Playbook

In [3]:
class BasePlay: 
    """
    Parent class for all sub classes of Playbook
    """

    def __init__(self, manager): 
        # self.manager takes over all properties inside PlaybookManager class
        self.manager = manager

    def run(self):
        # In case the run method of sub-class run method, this handles the Not Implemented error message
        raise NotImplementedError
    

#### Hypothesis Class

In [4]:
class HypothesisPlay(BasePlay):
    def run(self):
        
        hypothesis = self.manager.hypothesis
        insight = f"Initial ICP hypothesis created: {hypothesis}"
        self.manager.update_state("insight", insight)

        return insight

#### Market Research

In [5]:
class MarketResearchPlay(BasePlay):
    def run(self):
        # Skip the detailed logic of how the confidence score was assigned for demo purposes
        confidence_score = 72

        # pass the key/value pair for the dynamic attribute handler (update_state)
        self.manager.update_state("condidence", confidence_score)

        return confidence_score

#### Outreach Crafting

In [6]:
class OutreachCraftingPlay(BasePlay):
    def run(self):
        # Example pitch string
        pitch = "Helping fractional operators automate outreach workflows"

        self.manager.update_state("outreach_pitch", pitch)

        return pitch

#### Outreach Execution

In [7]:
class OutreachExecutionPlay(BasePlay):
    def run(self): 
        # Skip the detailed previous logic for feedback
        feedback = "Low response rate from startup founders"

        self.manager.update_state("feedback", feedback)

        return feedback

In [10]:
class HypothesisRefinementPlay(BasePlay):
    def run(self):
        feedback = self.manager.feedback

        if "low response" in feedback.lower(): 
            # Skip the detailed previous logic for refined_hypothesis
            refined_hypothesis = "Target agencies instead of startup founders"

            self.manager.update_state("hypothesis", refined_hypothesis)

        return refined_hypothesis 

#### Controller (Orchestration flow)

In [16]:
manager = PlaybookManager(hypothesis="Target startup founders")

plays = [
    HypothesisPlay(manager),
    MarketResearchPlay(manager),
    OutreachCraftingPlay(manager),
    OutreachExecutionPlay(manager),
    HypothesisRefinementPlay(manager)
]

#### Demo loop logic ####
max_loops = 3
current_loop = 0

while current_loop < max_loops:
    print(f"\n\n==== PLAYBOOK LOOP {current_loop + 1} ====")
    
    for play in plays:
        
        if isinstance(play, HypothesisRefinementPlay):
            print("\n====Final result of one loop execution cycle====")
        
        play.run()
    
        print(manager.get_shared_state())
    current_loop += 1



==== PLAYBOOK LOOP 1 ====
{'hypothesis': 'Target startup founders', 'insight': 'Initial ICP hypothesis created: Target startup founders', 'confidence': None, 'outreach_pitch': None, 'feedback': None}
{'hypothesis': 'Target startup founders', 'insight': 'Initial ICP hypothesis created: Target startup founders', 'confidence': None, 'outreach_pitch': None, 'feedback': None}
{'hypothesis': 'Target startup founders', 'insight': 'Initial ICP hypothesis created: Target startup founders', 'confidence': None, 'outreach_pitch': 'Helping fractional operators automate outreach workflows', 'feedback': None}
{'hypothesis': 'Target startup founders', 'insight': 'Initial ICP hypothesis created: Target startup founders', 'confidence': None, 'outreach_pitch': 'Helping fractional operators automate outreach workflows', 'feedback': 'Low response rate from startup founders'}

====Final result of one loop execution cycle====
{'hypothesis': 'Target agencies instead of startup founders', 'insight': 'Initial